In [1]:
from datasets import load_dataset
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
dataset = load_dataset(
    "coastalcph/lex_glue",
    "ledgar",
    cache_dir="../raw_data"
)

In [3]:
train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()

print(train_df.shape)
print(val_df.shape)

train_df.head()

(60000, 2)
(10000, 2)


,text,label
0,Except as otherwise set forth in this Debentur...,97
1,No ERISA Event has occurred or is reasonably e...,39
2,This Amendment may be executed by one or more ...,26
3,"From time to time, as and when required by the...",45
4,"Commencing March 7, 2016 and during the Employ...",11


# Full-label baseline

In [4]:
label_names = dataset["train"].features["label"].names

train_df["label_name"] = train_df["label"].apply(lambda x: label_names[x])
val_df["label_name"] = val_df["label"].apply(lambda x: label_names[x])

print("Training labels:", train_df["label_name"].nunique())
print("Validation labels:", val_df["label_name"].nunique())

Training labels: 100
Validation labels: 99


In [5]:
full_label_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            min_df=2,
            max_features=30000
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

In [6]:
X_train_full = train_df["text"]
y_train_full = train_df["label_name"]

X_val_full = val_df["text"]
y_val_full = val_df["label_name"]

print(X_train_full.shape)
print(y_train_full.shape)
print(X_val_full.shape)
print(y_val_full.shape)

(60000,)
(60000,)
(10000,)
(10000,)


In [7]:
full_label_pipeline.fit(X_train_full, y_train_full)

print("Training complete!")

Training complete!


In [8]:
y_val_pred_full = full_label_pipeline.predict(X_val_full)

full_accuracy = accuracy_score(y_val_full, y_val_pred_full)
full_macro_f1 = f1_score(y_val_full, y_val_pred_full, average="macro")
full_weighted_f1 = f1_score(
    y_val_full,
    y_val_pred_full,
    average="weighted"
)

print(f"Full-label validation accuracy: {full_accuracy:.4f}")
print(f"Full-label validation macro F1: {full_macro_f1:.4f}")
print(f"Full-label validation weighted F1: {full_weighted_f1:.4f}")

Full-label validation accuracy: 0.8327
Full-label validation macro F1: 0.7499
Full-label validation weighted F1: 0.8240


In [9]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(
    full_label_pipeline,
    "../models/ledgar_tfidf_logreg.joblib"
)

print("Saved to ../models/ledgar_tfidf_logreg.joblib")

Saved to ../models/ledgar_tfidf_logreg.joblib


## Linear SVM comparison

In [10]:
svm_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            min_df=2,
            max_features=30000
        )
    ),
    (
        "classifier",
        LinearSVC(random_state=42)
    )
])

svm_pipeline

,steps,"[('tfidf', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [11]:
svm_pipeline.fit(X_train_full, y_train_full)

print("Linear SVM training complete.")

Linear SVM training complete.


In [12]:
y_val_pred_svm = svm_pipeline.predict(X_val_full)

svm_accuracy = accuracy_score(y_val_full, y_val_pred_svm)
svm_macro_f1 = f1_score(
    y_val_full,
    y_val_pred_svm,
    average="macro"
)
svm_weighted_f1 = f1_score(
    y_val_full,
    y_val_pred_svm,
    average="weighted"
)

print(f"Linear SVM validation accuracy: {svm_accuracy:.4f}")
print(f"Linear SVM validation macro F1: {svm_macro_f1:.4f}")
print(f"Linear SVM validation weighted F1: {svm_weighted_f1:.4f}")

Linear SVM validation accuracy: 0.8507
Linear SVM validation macro F1: 0.7711
Linear SVM validation weighted F1: 0.8441


In [13]:
import joblib

joblib.dump(
    svm_pipeline,
    "../models/ledgar_tfidf_svm.joblib"
)

print("Saved to ../models/ledgar_tfidf_svm.joblib")

Saved to ../models/ledgar_tfidf_svm.joblib


In [14]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Linear SVM"
    ],
    "Accuracy": [
        full_accuracy,
        svm_accuracy
    ],
    "Macro F1": [
        full_macro_f1,
        svm_macro_f1
    ],
    "Weighted F1": [
        full_weighted_f1,
        svm_weighted_f1
    ]
})

comparison

,Model,Accuracy,Macro F1,Weighted F1
0,Logistic Regression,0.8327,0.749912,0.82402
1,Linear SVM,0.8507,0.771092,0.84410


# Optional comparison: Top-20 labels

In [15]:
top_20_labels = (
    train_df["label_name"]
    .value_counts()
    .head(20)
    .index
    .tolist()
)

top_20_labels

['Governing Laws',
 'Notices',
 'Counterparts',
 'Entire Agreements',
 'Severability',
 'Survival',
 'Amendments',
 'Assignments',
 'Expenses',
 'Terms',
 'Terminations',
 'Insurances',
 'Taxes',
 'Litigations',
 'Confidentiality',
 'Further Assurances',
 'General',
 'Compliance With Laws',
 'Indemnifications',
 'Waivers']

In [16]:
train_top20 = train_df[
    train_df["label_name"].isin(top_20_labels)
].copy()

val_top20 = val_df[
    val_df["label_name"].isin(top_20_labels)
].copy()

print("Training samples:", len(train_top20))
print("Validation samples:", len(val_top20))
print("Training classes:", train_top20["label_name"].nunique())
print("Validation classes:", val_top20["label_name"].nunique())

Training samples: 28998
Validation samples: 4799
Training classes: 20
Validation classes: 20


In [17]:
X_train = train_top20["text"]
y_train = train_top20["label_name"]

X_val = val_top20["text"]
y_val = val_top20["label_name"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

X_train: (28998,)
y_train: (28998,)
X_val: (4799,)
y_val: (4799,)


In [18]:
baseline_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            min_df=2,
            max_features=30000
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

baseline_pipeline

,steps,"[('tfidf', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [19]:
baseline_pipeline.fit(X_train, y_train)

print("Training completed.")

Training completed.


In [20]:
y_val_pred = baseline_pipeline.predict(X_val)

accuracy = accuracy_score(y_val, y_val_pred)
macro_f1 = f1_score(y_val, y_val_pred, average="macro")
weighted_f1 = f1_score(y_val, y_val_pred, average="weighted")

print(f"Validation accuracy: {accuracy:.4f}")
print(f"Validation macro F1: {macro_f1:.4f}")
print(f"Validation weighted F1: {weighted_f1:.4f}")

Validation accuracy: 0.9542
Validation macro F1: 0.9436
Validation weighted F1: 0.9542


In [21]:
print(
    classification_report(
        y_val,
        y_val_pred,
        zero_division=0
    )
)

                      precision    recall  f1-score   support

          Amendments       0.92      0.94      0.93       257
         Assignments       0.97      0.98      0.97       208
Compliance With Laws       0.97      0.95      0.96       167
     Confidentiality       0.96      0.97      0.96       149
        Counterparts       1.00      0.99      0.99       429
   Entire Agreements       0.97      0.99      0.98       389
            Expenses       0.94      0.98      0.96       170
  Further Assurances       0.94      0.95      0.95       155
             General       0.70      0.73      0.72       150
      Governing Laws       0.98      0.98      0.98       494
    Indemnifications       0.97      0.96      0.97       140
          Insurances       0.98      0.99      0.98       204
         Litigations       0.98      0.98      0.98       166
             Notices       0.96      0.96      0.96       398
        Severability       0.99      0.98      0.99       332
       

## Notes

Baseline model completed.

Still to do:
- Decide on final dataset.
- Test another model.
- Improve performance if necessary.